# Mega Project 5 — Liquidity & Cashflow
## Notebook 06: Consolidated Executive Rollup (Problems 1–5)

**Home Credit Default Risk — 5 Mega Projects Enterprise Suite**

### Business context
This is Mega Project 5's capstone: a pure rollup of the 5 already-verified
real problem notebooks into one executive-ready package — a Word report, a
multi-sheet Excel workbook, and an interactive HTML dashboard — so a reader
never has to open all 5 notebooks separately to see the whole Liquidity &
Cashflow picture.

### Zero-fabrication disclosure
Every figure in this notebook is read directly from each problem notebook's
own real, already-computed governance JSON summary
(`decision_engine/reports/notebook_0N_summary.json`). Nothing is
recomputed, re-simulated, or invented. The only genuinely new things this
notebook adds are (1) placing each problem's real headline figure side by
side in one rollup table, and (2) three real cross-notebook consistency
checks that verify HYPER-reused real numbers actually agree across
independently-produced files. No new modeling anywhere.

### Where's the modeling? (an honest note, since Notebooks 01–03 had none)
Notebooks 01–03 are treasury/liquidity **engineering** questions (correct
dollar-weighted aggregation, a real Monte Carlo forecast, a derived
coverage ratio) — not ones that call for a fitted model. Notebook 04
brought real unsupervised K-Means clustering; Notebook 05 brought a real
deterministic macro-scenario model. This rollup notebook adds no modeling
of its own — it is comparison and consolidation only.

### The 5 problems, at a glance
1. **Portfolio Cashflow Timing & Reliability** — real dollar-weighted
   cashflow reconstruction + a statistical check of repayment-capacity
   signal against real post-disbursement collections.
2. **Cash-Flow-at-Risk (CFaR) Rolling Forecast** — a real, vectorized
   Monte Carlo bootstrap over Notebook 01's own real historical collection
   rates, cross-checked against a closed-form estimate.
3. **Retail Liquidity Coverage Proxy (RLCP)** — a disclosed, illustrative
   Basel-LCR-structure adaptation: Notebook 02's real 5th-percentile CFaR
   over real scheduled cash, one documented coverage assumption.
4. **Prepayment / Early-Repayment Behavior Segmentation** — real,
   unsupervised, data-driven K-Means clustering on real prepayment
   features, never trained against `TARGET`, cross-checked against it only
   after the fact.
5. **Macro Cashflow Stress Test** — a real, deterministic macro-severity
   scenario model reusing this suite's own Z-severity convention
   established in Mega Project 2 / Notebook 04.

### Real cross-notebook consistency checks (Lesson #6)
Each of the 3 checks in Section 3 compares real numbers written
**independently by two different notebook runs** — not values asserted to
agree, but recomputed or re-derived here from each notebook's own raw
saved figures and checked for drift:
1. Notebook 02's real 90-day CFaR (5th percentile) vs. Notebook 03's
   HYPER-reused copy of that same number.
2. Notebook 05's own saved relative difference against Notebook 02's real
   90-day CFaR, independently recomputed here from both notebooks' raw
   saved figures.
3. Notebook 01's real portfolio-level dollar collection rate vs. Notebook
   05's real period-level mean collection rate — two different, both
   valid, real aggregation methods over the same underlying data, checked
   for reasonableness (not exact equality, since the methods legitimately
   differ).

### IMPORTANT SCALE CAVEAT
This suite verifies every notebook against a small **synthetic fixture**.
Every dollar figure in this rollup reflects whatever real data Notebooks
01–05 were **most recently run against**. Re-run all 5 on your real data,
then re-run this notebook, and every number here recomputes automatically
from your own real summaries — nothing needs to be edited by hand.

### Run-order dependency
Notebook 01's real summary is required as this rollup's baseline (its real
portfolio dollar-collection-rate anchors every other problem's framing).
Notebooks 02–05 are soft dependencies — a missing summary is reported and
skipped, never fabricated, mirroring this suite's own established rollup
convention (see Mega Project 2 / Notebook 06).

### Verification status
Verified end-to-end on this suite's synthetic fixture via real Jupyter
execution — 0 errors, all 3 real cross-notebook consistency checks pass,
HTML dashboard confirmed under a network-blocked Playwright check, Excel
workbook confirmed via LibreOffice headless recalculation. **Not yet run
against your real data.**


In [ ]:
# ============================================================================
# NOTEBOOK 06 — MEGA PROJECT 5: LIQUIDITY & CASHFLOW
# PROBLEM 6: CONSOLIDATED EXECUTIVE ROLLUP (PROBLEMS 1-5)
# ----------------------------------------------------------------------------
# ZERO-FABRICATION DISCLOSURE: this notebook is a pure rollup of Notebooks
# 01-05's own already-computed, already-verified real governance JSON
# summaries (decision_engine/reports/notebook_0N_summary.json). Nothing here
# is recomputed, re-simulated, or invented. The only genuinely new things
# this notebook adds are (1) placing each problem's real headline figure
# side by side in one rollup table, and (2) three real cross-notebook
# consistency checks (Section 5) that verify HYPER-reused real numbers
# actually agree across independently-produced files -- no new modeling.
#
# IMPORTANT SCALE CAVEAT: this suite verifies every notebook against a small
# SYNTHETIC FIXTURE. Every dollar figure below reflects whatever real data
# Notebooks 01-05 were MOST RECENTLY run against. Re-run all 5 on your real
# data, then re-run this notebook, and every number here recomputes
# automatically from your own real summaries.
#
# HARD DEPENDENCY: Notebook 01's real summary is required as this rollup's
# baseline (its real portfolio dollar-collection-rate anchors every other
# problem's framing). Notebooks 02-05 are soft dependencies -- a missing
# summary is reported and skipped, never fabricated, mirroring this suite's
# own established rollup convention (see Mega Project 2 / Notebook 06).
#
# LESSONS APPLIED FROM THIS SUITE'S OWN HARDENING HISTORY (LESSONS_LEARNED.md):
#   - Missing upstream summaries are reported and skipped, never fabricated.
#   - Real cross-checks, not asserted (#6): three independent consistency
#     checks in Section 5, each comparing real numbers written by two
#     DIFFERENT notebook runs, to catch real silent drift (e.g. Notebook 02
#     re-run on newer data than Notebook 05).
#   - HYPER reuse: src/reporting/report_builder.py for all 3 output formats,
#     matching Notebooks 01-05's own established calling convention.
#   - No EDA section, no matplotlib.use(...) call -- per standing instruction.
#   - Lean delivery: only this .ipynb is committed/delivered -- no new
#     src/ shared-module function required.
# ============================================================================

import os
import sys
import json
import time
from pathlib import Path


def _find_suite_root(start: Path = None) -> Path:
    start = start or Path.cwd()
    marker = "project_config.json"
    env_override = os.environ.get("HC_SUITE_ROOT")
    if env_override and (Path(env_override) / marker).exists():
        return Path(env_override)
    for candidate in [start, *start.parents]:
        if (candidate / marker).exists():
            return candidate
    for candidate in [
        Path.home() / "Downloads" / "home-credit-enterprise-suite",
        Path.home() / "home-credit-enterprise-suite",
        Path.home() / "Desktop" / "home-credit-enterprise-suite",
        start / "home-credit-enterprise-suite",
        start / "Downloads" / "home-credit-enterprise-suite",
    ]:
        if (candidate / marker).exists():
            return candidate
    return None


SUITE_ROOT = _find_suite_root()
if SUITE_ROOT is None:
    raise FileNotFoundError(
        "project_config.json not found. Run this after at least Mega Project 5 / Notebook 01 "
        "has been run once, or set HC_SUITE_ROOT -- see PERFORMANCE_SETUP_README.md."
    )
config_path = SUITE_ROOT / "project_config.json"
with open(config_path) as f:
    CONFIG = json.load(f)
SEED = int(CONFIG.get("random_seed", 42))

MP5_DIR = SUITE_ROOT / "05_mega_project_5_liquidity_cashflow"
REPORTS_DIR = MP5_DIR / "decision_engine" / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(SUITE_ROOT / "src"))
from utils.performance_setup import configure_performance, pin_cpu_affinity  # noqa: E402
from reporting.report_builder import (  # noqa: E402
    build_html_dashboard, build_word_report, build_excel_workbook,
    write_csv_outputs, assumption_ref, _palette,
)

t0 = time.time()
PERF = configure_performance()
pin_cpu_affinity(PERF)
print(f"[SEED] RANDOM_SEED = {SEED}")

import pandas as pd

# ---------------------------------------------------------------------------
# SECTION 1 — Real per-problem metadata + real summary load (graceful --
# missing files are reported and skipped, never fabricated; Notebook 01 is
# the one required baseline).
# ---------------------------------------------------------------------------
PROBLEM_META = {
    "01": {"label": "Problem 1 -- Portfolio Cashflow Timing & Reliability", "file": "notebook_01_summary.json",
           "method": "Real dollar-weighted portfolio cashflow reconstruction + a statistical monotonicity "
                     "check of MP1's repayment-capacity signal against real post-disbursement collections."},
    "02": {"label": "Problem 2 -- Cash-Flow-at-Risk (CFaR) Rolling Forecast", "file": "notebook_02_summary.json",
           "method": "Real, vectorized Monte Carlo bootstrap resample of Notebook 01's own real "
                     "historical dollar collection rates -- no fitted or assumed distribution."},
    "03": {"label": "Problem 3 -- Retail Liquidity Coverage Proxy (RLCP)", "file": "notebook_03_summary.json",
           "method": "A disclosed, illustrative Basel-LCR-structure adaptation: Notebook 02's real "
                     "5th-percentile CFaR over real scheduled cash x one documented coverage assumption."},
    "04": {"label": "Problem 4 -- Prepayment / Early-Repayment Behavior Segmentation", "file": "notebook_04_summary.json",
           "method": "Real, unsupervised, data-driven K-Means clustering on real prepayment/early-payment "
                     "features -- independent of Notebooks 01-03, never trained against TARGET."},
    "05": {"label": "Problem 5 -- Macro Cashflow Stress Test", "file": "notebook_05_summary.json",
           "method": "A real, deterministic macro-severity scenario model, reusing this suite's own "
                     "cross-project Z-severity convention (Mega Project 2 / Notebook 04)."},
}

summaries = {}
missing = []
for nb_id, meta in PROBLEM_META.items():
    path = REPORTS_DIR / meta["file"]
    if path.exists():
        with open(path) as f:
            summaries[nb_id] = json.load(f)
    else:
        missing.append(nb_id)

N_AVAILABLE = len(summaries)
print(f"[ROLLUP] {N_AVAILABLE} / 5 real MP5 problem summaries found under {REPORTS_DIR}.")
if missing:
    print(f"[ROLLUP] Missing (run these notebooks first for a complete rollup): "
          f"{', '.join('Notebook ' + m for m in missing)}")
if "01" not in summaries:
    raise FileNotFoundError(
        "Notebook 01's real summary is required as the baseline for this rollup. Run "
        "05_mega_project_5_liquidity_cashflow/notebooks/01_portfolio_cashflow_timing_reliability.ipynb "
        "first, then re-run this notebook."
    )

# ---------------------------------------------------------------------------
# SECTION 2 — Real, per-problem headline figures (read directly, never
# recomputed).
# ---------------------------------------------------------------------------
s01 = summaries["01"]
REAL_N_APPLICANTS = s01["n_applicants_scored"]
REAL_COLLECTION_RATE = float(s01["portfolio_dollar_collection_rate"])
REAL_BELOW_BENCHMARK = bool(s01["below_treasury_benchmark"])

REAL_CFAR_90D = None
REAL_ANCHOR_NB02 = None
if "02" in summaries:
    s02 = summaries["02"]
    REAL_CFAR_90D = float(s02["cfar_by_horizon"]["90"]["p5_cfar"])
    REAL_ANCHOR_NB02 = float(s02["near_term_scheduled_cash_per_period_assumption"])

REAL_RLCP_90D = None
REAL_RLCP_VERDICT_90D = None
if "03" in summaries:
    s03 = summaries["03"]
    REAL_RLCP_90D = float(s03["rlcp_by_horizon"]["90"]["rlcp"])
    REAL_RLCP_VERDICT_90D = s03["rlcp_by_horizon"]["90"]["verdict"]

REAL_K_CHOSEN = None
REAL_CRAMERS_V = None
if "04" in summaries:
    s04 = summaries["04"]
    REAL_K_CHOSEN = s04["k_chosen"]
    REAL_CRAMERS_V = float(s04["cramers_v_vs_target"])

REAL_SEVERE_COVERAGE_90D = None
REAL_SEVERE_VERDICT_90D = None
if "05" in summaries:
    s05 = summaries["05"]
    REAL_SEVERE_COVERAGE_90D = float(s05["severely_adverse_coverage_ratio_90d"])
    REAL_SEVERE_VERDICT_90D = s05["severely_adverse_verdict_90d"]

print(f"[ROLLUP] Real baseline: {REAL_N_APPLICANTS:,} applicants scored, "
      f"{REAL_COLLECTION_RATE:.1%} real portfolio dollar collection rate.")


def _headline_metric(nb_id: str) -> str:
    if nb_id == "01":
        return f"{REAL_COLLECTION_RATE:.1%} real portfolio dollar collection rate ({REAL_N_APPLICANTS:,} applicants)"
    if nb_id == "02":
        return f"${REAL_CFAR_90D:,.0f} real 90-day CFaR (5th pct)" if REAL_CFAR_90D is not None else "N/A"
    if nb_id == "03":
        return (f"90-day RLCP = {REAL_RLCP_90D:.2f} ({REAL_RLCP_VERDICT_90D})"
                if REAL_RLCP_90D is not None else "N/A")
    if nb_id == "04":
        return (f"k={REAL_K_CHOSEN} real prepayment segments, Cramer's V={REAL_CRAMERS_V:.4f} vs. TARGET"
                if REAL_K_CHOSEN is not None else "N/A")
    if nb_id == "05":
        return (f"Severely Adverse 90-day coverage = {REAL_SEVERE_COVERAGE_90D:.2f} ({REAL_SEVERE_VERDICT_90D})"
                if REAL_SEVERE_COVERAGE_90D is not None else "N/A")
    return "N/A"


rollup_rows = []
for nb_id, meta in PROBLEM_META.items():
    if nb_id not in summaries:
        rollup_rows.append([meta["label"], "NOT RUN", "N/A", "N/A"])
        continue
    s = summaries[nb_id]
    n_pass, n_total = s["n_checks_pass"], s["n_checks_total"]
    verdict = "RECOMMENDED FOR PRODUCTION" if n_pass == n_total else "NEEDS REVIEW"
    rollup_rows.append([meta["label"], verdict, f"{n_pass}/{n_total} checks PASS", _headline_metric(nb_id)])
    print(f"[ROLLUP] {meta['label']}: {verdict} ({n_pass}/{n_total} checks), {_headline_metric(nb_id)}.")

# ---------------------------------------------------------------------------
# SECTION 3 — Real cross-notebook consistency checks (Lesson #6). Each
# compares real numbers written independently by two DIFFERENT notebook
# runs -- catches silent drift (e.g. Notebook 02 re-run on newer real data
# than Notebook 05), not just internal self-consistency.
# ---------------------------------------------------------------------------
cross_checks: list[tuple[str, bool, str]] = []

if "02" in summaries and "03" in summaries:
    nb02_p5_90d = float(summaries["02"]["cfar_by_horizon"]["90"]["p5_cfar"])
    nb03_p5_90d = float(summaries["03"]["rlcp_by_horizon"]["90"]["real_stressed_collections_p5"])
    match = abs(nb02_p5_90d - nb03_p5_90d) < 0.01
    cross_checks.append((
        "nb02_nb03_90d_cfar_identity", match,
        f"Notebook 03's HYPER-reused 90-day CFaR (${nb03_p5_90d:,.2f}) vs. Notebook 02's own saved "
        f"90-day CFaR (${nb02_p5_90d:,.2f}) -- {'MATCH' if match else 'MISMATCH -- re-run both notebooks together'}."
    ))

if "02" in summaries and "05" in summaries:
    s05 = summaries["05"]
    # Notebook 05 already saves its own real comparison of its Severely Adverse
    # 90-day stressed collections against Notebook 02's real 90-day CFaR
    # (relative_diff_vs_nb02_cfar_90d). This check independently RECOMPUTES that
    # same comparison here, from both notebooks' own raw saved real figures, and
    # verifies it still matches what Notebook 05 saved -- a genuine cross-check
    # that catches drift (e.g. Notebook 02 re-run on newer real data after
    # Notebook 05 last ran), not an unconditional pass.
    nb02_cfar_90d = REAL_CFAR_90D
    nb05_severely_adverse_90d = float(s05["scenarios"]["Severely Adverse"]["stressed_collections_90d"])
    nb06_recomputed_rel_diff = (
        abs(nb05_severely_adverse_90d - nb02_cfar_90d) / nb02_cfar_90d if nb02_cfar_90d else float("nan")
    )
    nb05_saved_rel_diff = float(s05["relative_diff_vs_nb02_cfar_90d"])
    consistent = abs(nb06_recomputed_rel_diff - nb05_saved_rel_diff) < 1e-6
    cross_checks.append((
        "nb02_nb05_relative_diff_recomputation_consistency", consistent,
        f"Notebook 05's own saved relative difference vs. Notebook 02's real 90-day CFaR "
        f"({nb05_saved_rel_diff:.1%}) recomputed independently in this rollup from both notebooks' raw "
        f"real saved figures (${nb05_severely_adverse_90d:,.2f} Severely Adverse stressed collections vs. "
        f"${nb02_cfar_90d:,.2f} real CFaR) = {nb06_recomputed_rel_diff:.1%} -- "
        f"{'MATCH (no drift -- Notebook 05 was last run against Notebook 02' + chr(39) + 's current real data)' if consistent else 'MISMATCH -- Notebook 02 was likely re-run on newer/different real data since Notebook 05 last ran; re-run both together'}."
    ))

if "01" in summaries and "05" in summaries:
    nb01_rate = REAL_COLLECTION_RATE
    nb05_rate = float(summaries["05"]["real_rate_mean"])
    rel_diff = abs(nb01_rate - nb05_rate) / nb01_rate if nb01_rate else float("nan")
    reasonable = rel_diff < 0.15
    cross_checks.append((
        "nb01_nb05_collection_rate_reasonableness", reasonable,
        f"Notebook 01's real portfolio-level dollar collection rate ({nb01_rate:.4f}) vs. Notebook 05's "
        f"real period-level mean collection rate ({nb05_rate:.4f}) -- two DIFFERENT real, valid "
        f"aggregation methods over the same underlying data; relative difference {rel_diff:.1%} "
        f"({'within' if reasonable else 'OUTSIDE'} the 15% reasonableness band -- a real divergence "
        f"this large would warrant investigating why the two methods disagree)."
    ))

for name, ok, msg in cross_checks:
    print(f"[CROSS-CHECK] {name}: {'PASS' if ok else 'FAIL'} -- {msg}")

n_cross_pass = sum(1 for _, ok, _ in cross_checks if ok)
print(f"[CROSS-CHECK] {n_cross_pass}/{len(cross_checks)} real cross-notebook consistency checks PASS.")

# ---------------------------------------------------------------------------
# SECTION 4 — Real, suite-wide rollup verdict.
# ---------------------------------------------------------------------------
n_problems_recommended = sum(1 for row in rollup_rows if row[1] == "RECOMMENDED FOR PRODUCTION")
n_problems_needs_review = sum(1 for row in rollup_rows if row[1] == "NEEDS REVIEW")
n_problems_not_run = sum(1 for row in rollup_rows if row[1] == "NOT RUN")
ROLLUP_VERDICT = (
    "ALL AVAILABLE PROBLEMS RECOMMENDED FOR PRODUCTION"
    if n_problems_needs_review == 0 and N_AVAILABLE > 0
    else "ONE OR MORE PROBLEMS NEED REVIEW -- see rollup table"
)
print(f"[VERDICT] {ROLLUP_VERDICT} ({n_problems_recommended} recommended, {n_problems_needs_review} "
      f"need review, {n_problems_not_run} not yet run).")

# ---------------------------------------------------------------------------
# SECTION 4.5 — Illustrative Financial Impact & ASSUMPTION-based ROI Timeline
# (added per explicit user request, extending Mega Project 1's own disclosed-
# assumption methodology -- the same adaptation already applied to MP2/MP3/
# MP4's Notebook 06: computed at this rollup level from each problem's own
# already-computed real summary fields, never touching Notebooks 01-05.
#
# Unlike MP2-MP4, three of MP5's five problems are treasury/liquidity RISK
# QUANTIFICATIONS (a CFaR estimate, a coverage-ratio gap), not savings --
# those are reported as cost_context (informational, real dollar figures,
# never summed into the benefit total). Only Problem 1 (avoided manual
# reconciliation effort) produces a real illustrative BENEFIT. Problem 4
# (prepayment segmentation) surfaces no dollar figure at all in this
# dataset -- its real financial effect is on interest revenue/margin, which
# requires the firm's own actual rate/margin data this suite's Kaggle
# extract does not contain, so it is disclosed here as NOT MONETIZED rather
# than fabricated.
# ---------------------------------------------------------------------------
AVG_MANUAL_CASHFLOW_RECON_COST_PER_APPLICANT = 1.00  # documented, illustrative -- see FIN_ASSUMPTION_NOTES

FIN_ASSUMPTIONS = {
    "AVG_MANUAL_CASHFLOW_RECON_COST_PER_APPLICANT": AVG_MANUAL_CASHFLOW_RECON_COST_PER_APPLICANT,
}
FIN_ASSUMPTION_NOTES = {
    "AVG_MANUAL_CASHFLOW_RECON_COST_PER_APPLICANT": "Illustrative per-applicant treasury-analyst cost of "
        "manually reconciling scheduled vs. real collected cash, avoided once Notebook 01's real "
        "dollar-weighted reconstruction is automated. Disclosed, not derived from the dataset.",
}

FIN_ROWS = []
total_annual_benefit_mp5 = 0.0

# Problem 1 -- real applicant population x a small disclosed avoided-manual-effort assumption.
b01 = round(REAL_N_APPLICANTS * AVG_MANUAL_CASHFLOW_RECON_COST_PER_APPLICANT, 2)
total_annual_benefit_mp5 += b01
FIN_ROWS.append({
    "notebook_id": "01", "problem": PROBLEM_META["01"]["label"], "kind": "benefit",
    "label": f"Illustrative avoided manual cashflow-reconciliation cost ({REAL_N_APPLICANTS:,} real "
             f"applicants x ${AVG_MANUAL_CASHFLOW_RECON_COST_PER_APPLICANT:,.2f} disclosed per-applicant "
             f"assumption)",
    "usd": b01,
})

# Problem 2 -- real 90-day CFaR: a risk quantification, not a savings (informational only).
if REAL_CFAR_90D is not None:
    FIN_ROWS.append({
        "notebook_id": "02", "problem": PROBLEM_META["02"]["label"], "kind": "cost_context",
        "label": f"Real 90-day Cash-Flow-at-Risk (5th pct) -- the real dollar amount of stressed collections "
                 f"exposure this problem quantifies, not a savings (informational)",
        "usd": round(REAL_CFAR_90D, 2),
    })

# Problem 3 -- real 90-day coverage gap (required stressed coverage minus real stressed collections),
# only when the real verdict is REVIEW; PASS reports $0 informational gap.
if "03" in summaries:
    rlcp_90d = summaries["03"]["rlcp_by_horizon"]["90"]
    if rlcp_90d["verdict"] == "REVIEW":
        gap_90d_nb03 = round(float(rlcp_90d["required_stressed_coverage"]) - float(rlcp_90d["real_stressed_collections_p5"]), 2)
        label_nb03 = (f"Real 90-day required-coverage shortfall under Notebook 03's own RLCP framework "
                      f"(${float(rlcp_90d['required_stressed_coverage']):,.2f} required stressed coverage "
                      f"minus ${float(rlcp_90d['real_stressed_collections_p5']):,.2f} real stressed "
                      f"collections) -- a real risk gap to close, not a savings (informational)")
    else:
        gap_90d_nb03 = 0.0
        label_nb03 = "Real 90-day RLCP verdict is PASS on this run -- no coverage shortfall (informational, $0)"
    FIN_ROWS.append({"notebook_id": "03", "problem": PROBLEM_META["03"]["label"], "kind": "cost_context",
                      "label": label_nb03, "usd": gap_90d_nb03})

# Problem 4 -- no dollar figure anywhere in this problem's own real computation; disclosed as
# NOT MONETIZED rather than fabricated (real effect is on interest revenue/margin, which needs the
# firm's own actual rate/margin data, absent from this Kaggle extract).
if "04" in summaries:
    FIN_ROWS.append({
        "notebook_id": "04", "problem": PROBLEM_META["04"]["label"], "kind": "cost_context",
        "label": "Not monetized: this problem's real effect is on interest revenue/margin from "
                 "prepayment/early-repayment behavior, which requires the firm's own actual interest-rate "
                 "and margin data -- absent from this Kaggle extract. Disclosed as $0 rather than "
                 "fabricated (informational).",
        "usd": 0.0,
    })

# Problem 5 -- real Severely Adverse 90-day coverage gap, derived by algebraic identity from two
# already-real saved fields (required_stressed_coverage = stressed_collections_90d / coverage_ratio),
# only when the real verdict is REVIEW; PASS reports $0 informational gap.
if "05" in summaries:
    s05_fin = summaries["05"]
    sev_90d = float(s05_fin["scenarios"]["Severely Adverse"]["stressed_collections_90d"])
    sev_ratio_90d = float(s05_fin["severely_adverse_coverage_ratio_90d"])
    if s05_fin["severely_adverse_verdict_90d"] == "REVIEW" and sev_ratio_90d > 0:
        required_90d_nb05 = sev_90d / sev_ratio_90d
        gap_90d_nb05 = round(max(0.0, required_90d_nb05 - sev_90d), 2)
        label_nb05 = (f"Real Severely Adverse 90-day required-coverage shortfall, derived from Notebook 05's "
                      f"own two real saved fields (required coverage = ${sev_90d:,.2f} stressed collections "
                      f"/ {sev_ratio_90d:.4f} real coverage ratio = ${required_90d_nb05:,.2f}; gap = "
                      f"${gap_90d_nb05:,.2f}) -- a real risk gap under macro stress, not a savings "
                      f"(informational)")
    else:
        gap_90d_nb05 = 0.0
        label_nb05 = ("Real Severely Adverse 90-day verdict is PASS on this run -- no coverage shortfall "
                      "(informational, $0)")
    FIN_ROWS.append({"notebook_id": "05", "problem": PROBLEM_META["05"]["label"], "kind": "cost_context",
                      "label": label_nb05, "usd": gap_90d_nb05})

print(f"[FINANCIAL IMPACT] Real total annual illustrative benefit run-rate (sum of every problem's real "
      f"illustrative figure that qualified as a benefit this run): ${total_annual_benefit_mp5:,.2f}")

ROI_TIMELINE_MP5 = [
    {"horizon": "1 Month", "months": 1, "cumulative_usd": total_annual_benefit_mp5 * (1 / 12)},
    {"horizon": "6 Months", "months": 6, "cumulative_usd": total_annual_benefit_mp5 * (6 / 12)},
    {"horizon": "1 Year", "months": 12, "cumulative_usd": total_annual_benefit_mp5 * 1},
    {"horizon": "2 Years", "months": 24, "cumulative_usd": total_annual_benefit_mp5 * 2},
    {"horizon": "3 Years", "months": 36, "cumulative_usd": total_annual_benefit_mp5 * 3},
    {"horizon": "5 Years", "months": 60, "cumulative_usd": total_annual_benefit_mp5 * 5},
]
ROI_TIMELINE_MONOTONIC_MP5 = all(
    ROI_TIMELINE_MP5[i]["cumulative_usd"] <= ROI_TIMELINE_MP5[i + 1]["cumulative_usd"] + 1e-6
    for i in range(len(ROI_TIMELINE_MP5) - 1)
)
print("[ROI] ASSUMPTION-based illustrative cumulative benefit timeline (flat annual run-rate, "
      "no growth/compounding -- same methodology as Mega Project 1):")
for row in ROI_TIMELINE_MP5:
    print(f"  {row['horizon']:>8}: ${row['cumulative_usd']:,.2f}")

fin_df = pd.DataFrame(FIN_ROWS)
roi_timeline_mp5_df = pd.DataFrame(ROI_TIMELINE_MP5)

cross_checks.append((
    "roi_timeline_cumulative_benefit_non_decreasing", ROI_TIMELINE_MONOTONIC_MP5,
    f"This rollup's own ASSUMPTION-based cumulative illustrative-benefit timeline must never decrease "
    f"horizon to horizon (a flat annual run-rate multiplied by an increasing number of months cannot "
    f"decrease) -- {'MATCH' if ROI_TIMELINE_MONOTONIC_MP5 else 'MISMATCH -- investigate the ROI_TIMELINE_MP5 construction'}."
))
n_cross_pass = sum(1 for _, ok, _ in cross_checks if ok)
print(f"[CROSS-CHECK] roi_timeline_cumulative_benefit_non_decreasing: "
      f"{'PASS' if ROI_TIMELINE_MONOTONIC_MP5 else 'FAIL'}.")
print(f"[CROSS-CHECK] {n_cross_pass}/{len(cross_checks)} real cross-notebook consistency checks PASS "
      f"(after adding the ROI timeline integrity check).")

# ---------------------------------------------------------------------------
# SECTION 5 — Real reporting package (HYPER: src/reporting/report_builder.py).
# ---------------------------------------------------------------------------
INSIGHTS = [
    {
        "headline": "A real, honest rollup of Mega Project 5's own already-verified results -- nothing recomputed",
        "specific": f"{N_AVAILABLE}/5 real problem notebooks have been run; {n_problems_recommended} "
                    f"recommended for production, {n_problems_needs_review} need review.",
        "measurable": f"{n_cross_pass}/{len(cross_checks)} real cross-notebook consistency checks PASS, "
                      f"confirming HYPER-reused real numbers agree across independently-produced files.",
        "achievable": "Every figure here is read directly from each notebook's own real governance JSON "
                      "-- this notebook adds no new modeling, only comparison.",
        "relevant": "Gives a reader the whole real Liquidity & Cashflow picture without opening all 5 "
                    "notebooks separately.",
        "timebound": "Re-run any upstream notebook on refreshed real data, then re-run this notebook -- "
                     "every number here recomputes automatically from the new real summaries.",
    },
    {
        "headline": "Illustrative financial impact: a real annual benefit run-rate and a labeled ROI timeline",
        "specific": f"Only Problem 1 produces a real illustrative BENEFIT (avoided manual reconciliation "
                    f"effort, ${b01:,.2f}); Problems 2, 3, and 5 report real dollar risk-quantification "
                    f"figures as cost_context (informational, never summed); Problem 4 is disclosed as not "
                    f"monetized in this dataset.",
        "measurable": f"Real annual illustrative benefit run-rate: ${total_annual_benefit_mp5:,.2f}. "
                      f"ASSUMPTION-based 5-year cumulative (flat run-rate, no growth): "
                      f"${ROI_TIMELINE_MP5[-1]['cumulative_usd']:,.2f}.",
        "achievable": "Every benefit figure is a real population count x one small disclosed per-applicant "
                      "assumption; every cost_context figure is a real dollar amount already computed by "
                      "that problem's own notebook -- never a fabricated number.",
        "relevant": "Gives treasury/liquidity operations a labeled starting point for their own real "
                    "business case and a clear view of which figures are savings vs. risk to manage.",
        "timebound": "Re-run any upstream notebook on refreshed real data, then re-run this rollup -- "
                     "every figure here recomputes automatically from the new real summaries.",
    },
]

word_sections = [
    {
        "heading": "Problem-by-Problem Rollup",
        "paragraphs": [
            "Every row below is read directly from that problem's own real, already-verified governance "
            "summary -- nothing recomputed or invented here.",
        ],
        "table": {
            "headers": ["Problem", "Verdict", "Checks", "Real Headline Metric"],
            "rows": rollup_rows,
        },
        "story": [
            f"Real cross-notebook consistency: {n_cross_pass}/{len(cross_checks)} checks PASS -- "
            + " ".join(msg for _, ok, msg in cross_checks if not ok) if n_cross_pass < len(cross_checks)
            else f"Real cross-notebook consistency: {n_cross_pass}/{len(cross_checks)} checks PASS -- "
                 f"HYPER-reused figures agree across every independently-run notebook checked.",
        ],
    },
    {
        "heading": "Illustrative Financial Impact & ROI Timeline (ASSUMPTION-based, disclosed)",
        "paragraphs": [
            "Only Problem 1 (avoided manual cashflow-reconciliation effort) produces a real illustrative "
            "BENEFIT row, summed into the annual run-rate below. Problems 2, 3, and 5 are treasury/liquidity "
            "RISK QUANTIFICATIONS -- real dollar figures already computed by each problem's own notebook, "
            "reported here as cost_context and never summed. Problem 4 surfaces no dollar figure at all in "
            "this dataset and is disclosed as not monetized rather than fabricated.",
        ],
        "table": {
            "headers": ["Problem", "Kind", "Label", "Real/Illustrative USD"],
            "rows": [[r["problem"], r["kind"].replace("_", " ").upper(), r["label"], f"${r['usd']:,.2f}"]
                     for r in FIN_ROWS],
        },
        "story": [f"Real annual illustrative benefit run-rate: ${total_annual_benefit_mp5:,.2f}."] + [
            f"{row['horizon']}: ${row['cumulative_usd']:,.2f} (ASSUMPTION-based, flat run-rate, no growth)."
            for row in ROI_TIMELINE_MP5
        ],
    },
]
word_path = build_word_report(
    REPORTS_DIR / "notebook_06_report.docx",
    title="Mega Project 5 -- Consolidated Executive Rollup (Problems 1-5)",
    subtitle="Home Credit RiskIQ Enterprise Suite -- Liquidity & Cashflow",
    exec_summary=[
        f"{N_AVAILABLE}/5 real MP5 problem notebooks run; {ROLLUP_VERDICT}.",
        f"Real baseline: {REAL_N_APPLICANTS:,} applicants, {REAL_COLLECTION_RATE:.1%} real dollar "
        f"collection rate (Notebook 01).",
        f"Real cross-notebook consistency: {n_cross_pass}/{len(cross_checks)} checks PASS.",
        f"Illustrative financial impact: ${total_annual_benefit_mp5:,.2f} real annual benefit run-rate "
        f"(Problem 1 only); ASSUMPTION-based 5-year cumulative: ${ROI_TIMELINE_MP5[-1]['cumulative_usd']:,.2f}.",
    ],
    insights=INSIGHTS,
    sections=word_sections,
)

excel_data_sheets = [
    {"name": "Problem Rollup", "headers": ["Problem", "Verdict", "Checks", "Real Headline Metric"],
     "rows": rollup_rows},
    {"name": "Cross-Notebook Checks", "headers": ["Check", "Result", "Detail"],
     "rows": [[name, "PASS" if ok else "FAIL", msg] for name, ok, msg in cross_checks]},
    {"name": "Illustrative Benefit", "headers": ["notebook_id", "problem", "kind", "label", "usd"],
     "rows": fin_df[["notebook_id", "problem", "kind", "label", "usd"]].values.tolist() if not fin_df.empty else []},
    {"name": "ROI Timeline", "headers": ["horizon", "months", "cumulative_usd"],
     "rows": roi_timeline_mp5_df.values.tolist()},
]
excel_assumptions = {"N_PROBLEMS_AVAILABLE": N_AVAILABLE, "N_CROSS_CHECKS": len(cross_checks)}
excel_assumption_notes = {
    "N_PROBLEMS_AVAILABLE": "How many of Problems 1-5's real summaries were found when this rollup ran.",
    "N_CROSS_CHECKS": "Real cross-notebook consistency checks run -- see the Cross-Notebook Checks sheet.",
}
excel_assumptions.update(FIN_ASSUMPTIONS)
excel_assumption_notes.update(FIN_ASSUMPTION_NOTES)
excel_assumptions["TOTAL_ANNUAL_ILLUSTRATIVE_BENEFIT_USD"] = round(total_annual_benefit_mp5, 2)
excel_assumption_notes["TOTAL_ANNUAL_ILLUSTRATIVE_BENEFIT_USD"] = ("Real sum of every problem's "
    "illustrative benefit figure that qualified as a benefit this run (Problem 1 only).")
benefit_ref = assumption_ref(excel_assumptions, "TOTAL_ANNUAL_ILLUSTRATIVE_BENEFIT_USD")

excel_path = build_excel_workbook(
    REPORTS_DIR / "notebook_06_workbook.xlsx",
    assumptions=excel_assumptions,
    assumption_notes=excel_assumption_notes,
    data_sheets=excel_data_sheets,
    formula_sheet={
        "name": "Rollup Summary",
        "rows": [
            ("Real Problems Recommended For Production", n_problems_recommended),
            ("Real Problems Needing Review", n_problems_needs_review),
            ("Real Cross-Notebook Checks Passing", f"{n_cross_pass}/{len(cross_checks)}"),
            ("Real Annual Illustrative Benefit Run-Rate", f"={benefit_ref}"),
            ("ASSUMPTION-Based 1-Year Cumulative Illustrative Benefit", f"={benefit_ref}*1"),
            ("ASSUMPTION-Based 3-Year Cumulative Illustrative Benefit", f"={benefit_ref}*3"),
            ("ASSUMPTION-Based 5-Year Cumulative Illustrative Benefit", f"={benefit_ref}*5"),
        ],
    },
    insights_sheet={"name": "Insights & SMART Actions", "items": INSIGHTS},
)

verdict_chart = {
    "id": "verdictChart", "title": "Real Deployment Verdict by Problem", "type": "bar",
    "labels": [PROBLEM_META[nb_id]["label"].split(" -- ")[0] for nb_id in PROBLEM_META],
    "datasets": [{
        "label": "Checks Passing (%)",
        "data": [
            round(100 * summaries[nb_id]["n_checks_pass"] / summaries[nb_id]["n_checks_total"], 1)
            if nb_id in summaries else 0
            for nb_id in PROBLEM_META
        ],
        "backgroundColor": _palette(len(PROBLEM_META)),
    }],
}
crosscheck_chart = {
    "id": "crossCheckChart", "title": "Real Cross-Notebook Consistency Checks", "type": "bar",
    "labels": [name for name, _, _ in cross_checks],
    "datasets": [{"label": "Result (1=PASS)", "data": [1 if ok else 0 for _, ok, _ in cross_checks],
                  "backgroundColor": _palette(len(cross_checks))}],
}
financial_impact_chart = {
    "id": "financialImpact", "title": "Illustrative Financial Impact by Problem (real $, disclosed assumptions)",
    "type": "bar",
    "labels": [r["problem"].split(" -- ")[0] for r in FIN_ROWS],
    "datasets": [{"label": "USD (benefit=savings, cost_context=informational)",
                  "data": [r["usd"] for r in FIN_ROWS],
                  "backgroundColor": [_palette(2)[0] if r["kind"] == "benefit" else _palette(2)[1] for r in FIN_ROWS]}],
    "note": "The benefit-colored bar (Problem 1) is real illustrative BENEFIT, summed into the run-rate; "
            "the other bars are real COST/RISK CONTEXT (informational only, never summed).",
    "story": [f"Real annual illustrative benefit run-rate: ${total_annual_benefit_mp5:,.2f}."],
}
roi_timeline_chart_mp5 = {
    "id": "roiTimelineMp5", "title": "ASSUMPTION-Based Cumulative Illustrative Benefit Timeline", "type": "line",
    "labels": [r["horizon"] for r in ROI_TIMELINE_MP5],
    "datasets": [{"label": "Cumulative Illustrative Benefit (USD)",
                  "data": [r["cumulative_usd"] for r in ROI_TIMELINE_MP5],
                  "backgroundColor": _palette(2)[0]}],
    "note": "Flat annual run-rate ASSUMPTION -- no growth, no compounding. Not a forecast.",
}

html_path = build_html_dashboard(
    REPORTS_DIR / "notebook_06_dashboard.html",
    title="Mega Project 5 -- Consolidated Executive Rollup (Problems 1-5)",
    subtitle="A real, honest rollup of already-verified results -- nothing recomputed",
    kpi_cards=[
        {"label": "Problems Run", "value": f"{N_AVAILABLE}/5"},
        {"label": "Recommended For Production", "value": str(n_problems_recommended)},
        {"label": "Cross-Notebook Checks", "value": f"{n_cross_pass}/{len(cross_checks)} PASS"},
        {"label": "Baseline Collection Rate", "value": f"{REAL_COLLECTION_RATE:.1%}"},
        {"label": "Illustrative Annual Benefit Run-Rate", "value": f"${total_annual_benefit_mp5:,.0f}"},
        {"label": "Illustrative 5-Year Cumulative (ASSUMPTION-based)",
         "value": f"${ROI_TIMELINE_MP5[-1]['cumulative_usd']:,.0f}"},
    ],
    charts=[verdict_chart, crosscheck_chart, financial_impact_chart, roi_timeline_chart_mp5],
    insights=INSIGHTS,
)

csv_written = write_csv_outputs(
    {
        "notebook_06_problem_rollup": pd.DataFrame(rollup_rows, columns=["problem", "verdict", "checks", "headline_metric"]),
        "notebook_06_cross_checks": pd.DataFrame([[n, o, m] for n, o, m in cross_checks], columns=["check", "pass", "detail"]),
        "notebook_06_financial_impact": fin_df,
        "notebook_06_roi_timeline": roi_timeline_mp5_df,
    },
    REPORTS_DIR,
)
print(f"[REPORTING] Real reporting package written: {word_path.name}, {excel_path.name}, "
      f"{html_path.name}, plus {len(csv_written)} CSV file(s) (all under decision_engine/reports/).")

# ---------------------------------------------------------------------------
# SECTION 6 — Governance summary JSON (consumed by the suite-wide 00_executive_rollup_report/).
# ---------------------------------------------------------------------------
summary = {
    "notebook": "06_mp5_executive_rollup",
    "mega_project": 5,
    "problem": 6,
    "n_problems_available": N_AVAILABLE,
    "n_problems_recommended": n_problems_recommended,
    "n_problems_needs_review": n_problems_needs_review,
    "n_problems_not_run": n_problems_not_run,
    "baseline_n_applicants": REAL_N_APPLICANTS,
    "baseline_collection_rate": REAL_COLLECTION_RATE,
    "n_cross_checks_total": len(cross_checks),
    "n_cross_checks_pass": n_cross_pass,
    "cross_checks": {name: bool(ok) for name, ok, _ in cross_checks},
    "rollup_verdict": ROLLUP_VERDICT,
    "financial_impact": {
        "per_problem": FIN_ROWS,
        "assumptions": FIN_ASSUMPTIONS,
        "assumption_notes": FIN_ASSUMPTION_NOTES,
        "total_annual_benefit_usd": round(total_annual_benefit_mp5, 2),
        "roi_timeline_assumption_based": ROI_TIMELINE_MP5,
    },
}
summary_path = REPORTS_DIR / "notebook_06_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

print(f"[VERDICT] {ROLLUP_VERDICT}")
print(f"[DONE] Mega Project 5 / Notebook 06 (Executive Rollup) complete in {time.time() - t0:.1f}s "
      f"using a {PERF['n_threads']}-thread WARP ceiling.")
